In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
valuation_date = '2025-11-14'
db_path = 'arms_database.db'

from datetime import datetime as _dt
_vd = _dt.strptime(valuation_date, '%Y-%m-%d')
_year_folder = _vd.strftime('%Y')
_month_folder = f'{_vd.month}.{_vd.strftime("%B")}'
output_file_name = f'Price Attribution {valuation_date}.xlsx'
output_file_path = rf'P:\Application\Risk Mgmt\MRM\ARMS\output\{output_file_name}'

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# ── Get previous business day ─────────────────────────────────────────────────
def get_prev_business_day(date_str):
    d = datetime.strptime(date_str, '%Y-%m-%d')
    d -= timedelta(days=1)
    while d.weekday() >= 5:  # skip Saturday=5, Sunday=6
        d -= timedelta(days=1)
    return d.strftime('%Y-%m-%d')

prev_date = get_prev_business_day(valuation_date)
print(f'Attribution period:  {prev_date}  →  {valuation_date}')

In [ ]:
# ── Load bond metrics for both dates (long → wide pivot) ─────────────────────
# dv01  : price change per 1bp parallel yield shift (in price % units, per bond)
# cr01  : price change per 1bp credit spread shift (in price % units, per bond)
# par_yield_rf_rate_pp : risk-free rate used in pricing (in % e.g. 4.5 = 4.5%)
# clean_price_hybrid   : ARMS calculated clean price (% of par)
# dtm   : days to maturity
# ytm_dec : yield to maturity in decimal (e.g. 0.045)

METRICS_NEEDED = (
    "'clean_price_hybrid', 'clean_price_adj', 'mod_duration', "
    "'dv01', 'cr01', 'par_yield_rf_rate_pp', 'dtm', 'ytm_dec', 'accrued_interest'"
)

METRICS_QUERY = f"""
    SELECT RepDate, isin, metric, value
    FROM bond_metrics
    WHERE RepDate IN ('{prev_date}', '{valuation_date}')
      AND metric IN ({METRICS_NEEDED})
"""

with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True, timeout=10) as conn:
    df_metrics_long = pd.read_sql_query(METRICS_QUERY, conn)

# Pivot to wide: rows = (RepDate, isin), columns = metric values
df_metrics = df_metrics_long.pivot_table(
    index=['RepDate', 'isin'], columns='metric', values='value'
).reset_index()
df_metrics.columns.name = None

print(f'Metrics loaded: {df_metrics.shape[0]} rows, dates: {df_metrics.RepDate.unique()}')
df_metrics.head(3)

In [ ]:
# ── Load bond_price for both dates ────────────────────────────────────────────
PRICE_QUERY = f"""
    SELECT RepDate, isin, pricing_source, price_bid, price_last,
           ytm_bid, oas_spread, z_spread, factor_principal
    FROM bond_price
    WHERE RepDate IN ('{prev_date}', '{valuation_date}')
"""

with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True, timeout=10) as conn:
    df_price = pd.read_sql_query(PRICE_QUERY, conn)

print(f'Prices loaded: {df_price.shape[0]} rows')
df_price.head(3)

In [ ]:
# ── Load bond dictionary (metadata) ──────────────────────────────────────────
BONDS_QUERY = """
    SELECT db.isin, db.bond_name, db.class_internal, db.pricing_source,
           db.maturity, db.coupon_type, db.coupon, db.par_value,
           db.coupon_frequency, db.redemption_type,
           di.company_name AS issuer
    FROM dic_bonds db
    LEFT JOIN dic_issuers di ON db.bloom_company_id = di.bloom_company_id
"""

with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True, timeout=10) as conn:
    df_bonds = pd.read_sql_query(BONDS_QUERY, conn)

print(f'Bond dictionary: {df_bonds.shape[0]} bonds')
df_bonds.head(3)

In [ ]:
# ── Load positions (for portfolio type, quantity, market value context) ───────
POS_QUERY = f"""
    SELECT isin, portfolio_type, SUM(quantity) AS quantity,
           SUM(total_notional) AS total_notional,
           SUM(invested_amount) AS invested_amount_fc,
           SUM(invested_amount_azn) AS invested_amount_azn,
           currency
    FROM positions
    WHERE RepDate = '{valuation_date}'
    GROUP BY isin, portfolio_type, currency
"""

with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True, timeout=10) as conn:
    df_pos = pd.read_sql_query(POS_QUERY, conn)

print(f'Positions loaded: {df_pos.shape[0]} rows')
df_pos.head(3)

In [ ]:
# ── Check for coupon / principal payments between prev_date and valuation_date ─
# A payment on this period reduces dirty price (accrued interest resets)
CF_QUERY = f"""
    SELECT isin, cashflows_date, coupon, principal
    FROM dic_bond_cf
    WHERE cashflows_date > '{prev_date}'
      AND cashflows_date <= '{valuation_date}'
"""

with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True, timeout=10) as conn:
    df_cf = pd.read_sql_query(CF_QUERY, conn)

# Aggregate: total coupon and principal paid per ISIN in this period
if not df_cf.empty:
    df_cf_agg = df_cf.groupby('isin').agg(
        coupon_paid=('coupon', 'sum'),
        principal_paid=('principal', 'sum'),
        payment_dates=('cashflows_date', lambda x: ', '.join(x.astype(str)))
    ).reset_index()
    print(f'Payments found between {prev_date} and {valuation_date}:')
    display(df_cf_agg)
else:
    df_cf_agg = pd.DataFrame(columns=['isin', 'coupon_paid', 'principal_paid', 'payment_dates'])
    print(f'No coupon or principal payments between {prev_date} and {valuation_date}')

In [ ]:
# ── Build attribution table ───────────────────────────────────────────────────
#
# Split metrics into prev and curr, then merge side-by-side
#
# COMPONENTS:
#   1. rate_effect    = -dv01 × Δrf_bps
#      where Δrf_bps = (par_yield_rf_rate_pp_t - par_yield_rf_rate_pp_t-1) × 100
#      (par_yield_rf_rate_pp is in %, so ×100 converts to bps)
#
#   2. spread_effect  = -cr01 × Δoas_bps
#      where Δoas_bps = oas_spread_t - oas_spread_t-1   (already in bps)
#
#   3. pull_to_par    = (100 - clean_price_prev) / dtm_prev
#      The natural daily drift toward par as maturity approaches
#      Positive for discount bonds, negative for premium bonds
#
#   4. residual       = total_price_change - rate_effect - spread_effect - pull_to_par
#      Captures anything not explained: convexity, cross-effects, model error

# --- Split metrics into prev/curr ---
m_prev = df_metrics[df_metrics['RepDate'] == prev_date].copy()
m_curr = df_metrics[df_metrics['RepDate'] == valuation_date].copy()

p_prev = df_price[df_price['RepDate'] == prev_date].copy()
p_curr = df_price[df_price['RepDate'] == valuation_date].copy()

# --- Merge metrics prev + curr on isin ---
m_cols = ['isin', 'clean_price_hybrid', 'clean_price_adj', 'mod_duration',
          'dv01', 'cr01', 'par_yield_rf_rate_pp', 'dtm', 'ytm_dec', 'accrued_interest']
m_prev = m_prev[[c for c in m_cols if c in m_prev.columns]]
m_curr = m_curr[[c for c in m_cols if c in m_curr.columns]]

df = m_prev.merge(m_curr, on='isin', suffixes=('_prev', '_curr'), how='inner')

# --- Merge bond_price prev + curr ---
p_prev = p_prev[['isin', 'oas_spread', 'ytm_bid', 'factor_principal']].rename(
    columns={'oas_spread': 'oas_prev', 'ytm_bid': 'ytm_prev', 'factor_principal': 'factor_prev'})
p_curr = p_curr[['isin', 'oas_spread', 'ytm_bid', 'factor_principal', 'pricing_source']].rename(
    columns={'oas_spread': 'oas_curr', 'ytm_bid': 'ytm_curr', 'factor_principal': 'factor_curr'})

df = df.merge(p_prev, on='isin', how='left')
df = df.merge(p_curr, on='isin', how='left')

# --- Merge bond metadata ---
df = df.merge(df_bonds[['isin', 'bond_name', 'class_internal', 'issuer',
                         'maturity', 'coupon', 'coupon_type', 'par_value',
                         'redemption_type', 'pricing_source']], on='isin', how='left')

# --- Merge position info (portfolio_type, currency) ---
df = df.merge(df_pos[['isin', 'portfolio_type', 'currency', 'quantity',
                       'total_notional', 'invested_amount_fc']],
              on='isin', how='left')

# --- Merge payment info ---
df = df.merge(df_cf_agg[['isin', 'coupon_paid', 'principal_paid', 'payment_dates']],
              on='isin', how='left')
df['coupon_paid'] = df['coupon_paid'].fillna(0)
df['principal_paid'] = df['principal_paid'].fillna(0)
df['payment_dates'] = df['payment_dates'].fillna('')

print(f'Attribution table: {df.shape[0]} bonds')
df.head(3)

In [ ]:
# ── Calculate attribution components ─────────────────────────────────────────

# 1. Total price change (in % of par)
df['price_prev']   = df['clean_price_hybrid_prev']
df['price_curr']   = df['clean_price_hybrid_curr']
df['price_change'] = df['price_curr'] - df['price_prev']

# 2. Rate effect: -dv01 × Δrf_bps
#    par_yield_rf_rate_pp is in %  →  ×100 to convert to basis points
df['delta_rf_pp']  = df['par_yield_rf_rate_pp_curr'] - df['par_yield_rf_rate_pp_prev']
df['delta_rf_bps'] = df['delta_rf_pp'] * 100
df['rate_effect']  = -df['dv01_prev'] * df['delta_rf_bps']

# 3. Spread effect: -cr01 × Δoas_bps
#    oas_spread is already in basis points
df['delta_oas_bps']  = df['oas_curr'] - df['oas_prev']
df['spread_effect']  = -df['cr01_prev'] * df['delta_oas_bps']

# 4. Pull to par: (100 - price_prev) / dtm_prev  (per 1 day elapsed)
#    Positive = bond was below par → drifts up toward 100
#    Negative = bond was above par → drifts down toward 100
#    Zero coupon bonds: this is the dominant driver day-to-day
df['pull_to_par'] = np.where(
    df['dtm_prev'] > 0,
    (100 - df['price_prev']) / df['dtm_prev'],
    0
)

# 5. Residual: what is not explained by the three components above
df['rate_effect']   = df['rate_effect'].fillna(0)
df['spread_effect'] = df['spread_effect'].fillna(0)
df['residual'] = (df['price_change']
                  - df['rate_effect']
                  - df['spread_effect']
                  - df['pull_to_par'])

# 6. Attribution check: sum should equal price_change
df['check'] = (df['rate_effect'] + df['spread_effect']
               + df['pull_to_par'] + df['residual'] - df['price_change']).round(6)

print('Attribution checks (all should be ~0):')
print(df['check'].describe())

In [ ]:
# ── Display summary table ─────────────────────────────────────────────────────
display_cols = [
    'isin', 'issuer', 'portfolio_type', 'currency', 'pricing_source',
    'price_prev', 'price_curr', 'price_change',
    'rate_effect', 'spread_effect', 'pull_to_par', 'residual',
    'delta_rf_bps', 'delta_oas_bps',
    'coupon_paid', 'principal_paid', 'payment_dates',
    'dtm_prev', 'mod_duration_prev'
]
display_cols = [c for c in display_cols if c in df.columns]

df_display = df[display_cols].copy()
df_display = df_display.sort_values(['portfolio_type', 'price_change'])

# Format numbers
for col in ['price_prev', 'price_curr', 'price_change',
            'rate_effect', 'spread_effect', 'pull_to_par', 'residual']:
    if col in df_display.columns:
        df_display[col] = df_display[col].round(4)

display(df_display)

In [ ]:
# ── Portfolio-level summary ───────────────────────────────────────────────────
df_summary = df.groupby('portfolio_type').agg(
    bond_count=('isin', 'count'),
    avg_price_prev=('price_prev', 'mean'),
    avg_price_curr=('price_curr', 'mean'),
    avg_price_change=('price_change', 'mean'),
    avg_rate_effect=('rate_effect', 'mean'),
    avg_spread_effect=('spread_effect', 'mean'),
    avg_pull_to_par=('pull_to_par', 'mean'),
    avg_residual=('residual', 'mean'),
).round(4).reset_index()

print(f'\nPortfolio summary  ({prev_date} → {valuation_date}):')
display(df_summary)

In [ ]:
# ── Attribution waterfall chart (portfolio-level) ─────────────────────────────
#
# Shows: for each portfolio type, how much of the average price change
# came from each driver.

components = ['rate_effect', 'spread_effect', 'pull_to_par', 'residual']
component_labels = ['Rate Effect', 'Spread Effect', 'Pull to Par', 'Residual']
component_colors = ['#ef4444', '#f97316', '#4ade80', '#94a3b8']

portfolio_types = df_summary['portfolio_type'].tolist()

fig = go.Figure()

for comp, label, color in zip(components, component_labels, component_colors):
    col_name = f'avg_{comp}'
    if col_name in df_summary.columns:
        values = df_summary[col_name].tolist()
        fig.add_trace(go.Bar(
            name=label,
            x=portfolio_types,
            y=values,
            marker_color=color,
            text=[f'{v:+.4f}' for v in values],
            textposition='outside',
            textfont=dict(size=11)
        ))

fig.update_layout(
    title=dict(
        text=f'Price Change Attribution by Portfolio  |  {prev_date} → {valuation_date}',
        font=dict(size=16, color='#e2e8f0')
    ),
    barmode='group',
    template='plotly_dark',
    paper_bgcolor='#0a0e1a',
    plot_bgcolor='#111827',
    xaxis=dict(title='Portfolio', tickfont=dict(color='#94a3b8')),
    yaxis=dict(title='Avg Price Change (% of par)', tickfont=dict(color='#94a3b8'),
               gridcolor='#1e2534', zeroline=True, zerolinecolor='#475569'),
    legend=dict(font=dict(color='#94a3b8'), bgcolor='#111827', bordercolor='#1e2534'),
    margin=dict(l=60, r=40, t=80, b=60),
    height=500
)

fig.show()

In [ ]:
# ── Bond-level waterfall chart (top movers) ───────────────────────────────────
#
# Shows the 15 bonds with the largest absolute price change,
# with their decomposition.

top_n = 15
df_top = df.nlargest(top_n, 'price_change').copy()
df_top['label'] = df_top['isin'] + '\n' + df_top['issuer'].fillna('').str[:20]

fig2 = go.Figure()

for comp, label, color in zip(components, component_labels, component_colors):
    if comp in df_top.columns:
        fig2.add_trace(go.Bar(
            name=label,
            x=df_top['isin'],
            y=df_top[comp],
            marker_color=color
        ))

# Overlay: total price change line
fig2.add_trace(go.Scatter(
    name='Total Price Change',
    x=df_top['isin'],
    y=df_top['price_change'],
    mode='markers',
    marker=dict(color='#00b4d8', size=8, symbol='diamond'),
))

fig2.update_layout(
    title=dict(
        text=f'Top {top_n} Bonds by Price Increase  |  Attribution Breakdown',
        font=dict(size=15, color='#e2e8f0')
    ),
    barmode='stack',
    template='plotly_dark',
    paper_bgcolor='#0a0e1a',
    plot_bgcolor='#111827',
    xaxis=dict(title='ISIN', tickangle=-45, tickfont=dict(color='#94a3b8', size=10)),
    yaxis=dict(title='Price Change (% of par)', tickfont=dict(color='#94a3b8'),
               gridcolor='#1e2534', zeroline=True, zerolinecolor='#475569'),
    legend=dict(font=dict(color='#94a3b8'), bgcolor='#111827', bordercolor='#1e2534'),
    margin=dict(l=60, r=40, t=80, b=100),
    height=520
)

fig2.show()

In [ ]:
# ── Bond-level waterfall chart (top losers) ───────────────────────────────────
df_bot = df.nsmallest(top_n, 'price_change').copy()

fig3 = go.Figure()

for comp, label, color in zip(components, component_labels, component_colors):
    if comp in df_bot.columns:
        fig3.add_trace(go.Bar(
            name=label,
            x=df_bot['isin'],
            y=df_bot[comp],
            marker_color=color
        ))

fig3.add_trace(go.Scatter(
    name='Total Price Change',
    x=df_bot['isin'],
    y=df_bot['price_change'],
    mode='markers',
    marker=dict(color='#00b4d8', size=8, symbol='diamond'),
))

fig3.update_layout(
    title=dict(
        text=f'Top {top_n} Bonds by Price Decline  |  Attribution Breakdown',
        font=dict(size=15, color='#e2e8f0')
    ),
    barmode='stack',
    template='plotly_dark',
    paper_bgcolor='#0a0e1a',
    plot_bgcolor='#111827',
    xaxis=dict(title='ISIN', tickangle=-45, tickfont=dict(color='#94a3b8', size=10)),
    yaxis=dict(title='Price Change (% of par)', tickfont=dict(color='#94a3b8'),
               gridcolor='#1e2534', zeroline=True, zerolinecolor='#475569'),
    legend=dict(font=dict(color='#94a3b8'), bgcolor='#111827', bordercolor='#1e2534'),
    margin=dict(l=60, r=40, t=80, b=100),
    height=520
)

fig3.show()

In [ ]:
# ── Save to Excel ─────────────────────────────────────────────────────────────
import os
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

excel_cols = [
    'isin', 'issuer', 'bond_name', 'portfolio_type', 'currency',
    'pricing_source', 'class_internal', 'maturity', 'coupon',
    'price_prev', 'price_curr', 'price_change',
    'rate_effect', 'spread_effect', 'pull_to_par', 'residual',
    'delta_rf_bps', 'delta_oas_bps',
    'mod_duration_prev', 'dv01_prev', 'cr01_prev',
    'dtm_prev', 'ytm_dec_prev',
    'coupon_paid', 'principal_paid', 'payment_dates'
]
excel_cols = [c for c in excel_cols if c in df.columns]

df_excel = df[excel_cols].copy().sort_values(['portfolio_type', 'issuer'])

# Rename for readability
rename_map = {
    'price_prev':        f'Clean Price ({prev_date})',
    'price_curr':        f'Clean Price ({valuation_date})',
    'price_change':      'Price Change',
    'rate_effect':       'Rate Effect',
    'spread_effect':     'Spread Effect',
    'pull_to_par':       'Pull to Par',
    'residual':          'Residual',
    'delta_rf_bps':      'Δ Risk-Free Rate (bps)',
    'delta_oas_bps':     'Δ OAS Spread (bps)',
    'mod_duration_prev': 'Mod. Duration',
    'dv01_prev':         'DV01',
    'cr01_prev':         'CR01',
    'dtm_prev':          'Days to Maturity',
    'ytm_dec_prev':      'YTM',
    'coupon_paid':       'Coupon Paid',
    'principal_paid':    'Principal Paid',
    'payment_dates':     'Payment Dates'
}
df_excel = df_excel.rename(columns=rename_map)

with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    df_excel.to_excel(writer, sheet_name='Bond Attribution', index=False)
    df_summary.to_excel(writer, sheet_name='Portfolio Summary', index=False)

print(f'Saved: {output_file_path}')